In [0]:
%sql
-- Databricks notebook source
-- Pipeline  : rental_churn_query
-- Source    : Redshift proc public.rental_churn_query_proc() — migrated to Databricks native
-- Gold view : furlenco_gold.analytics.rental_churn_query
-- Schedule  : configured in Databricks Workflow (existing pipelines use 0 */4 * * *)
-- Swap      : zero-downtime atomic DEEP CLONE — live table is never dropped mid-refresh
-- Timezone  : all current_timestamp() outputs converted to IST (+ INTERVAL 330 MINUTES)

-- ============================================================================
-- BLOCK 1 — WRITE STAGING
-- Full refresh into staging. If this block fails, the live table is untouched.
-- ============================================================================

CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.rental_churn_query_staging AS
WITH
  base AS (
    SELECT
      items.id AS item_id,
      activation_date,
      return_items.state AS return_item_state,
      user_id,
      CASE
        WHEN return_items.created_at IS NULL
         AND rent_to_purchase_orders.created_at IS NULL THEN NULL
        WHEN rent_to_purchase_orders.created_at IS NULL THEN return_id
        WHEN return_items.created_at IS NULL THEN rent_to_purchase_order_id
      END AS Transaction_id,
      CASE
        WHEN return_items.created_at IS NULL
         AND rent_to_purchase_orders.created_at IS NULL THEN DATE_ADD(CURRENT_DATE(), 1)   -- [MIGRATED] CURRENT_DATE + 1 → DATE_ADD
        WHEN rent_to_purchase_orders.created_at IS NULL THEN return_items.created_at + INTERVAL 330 MINUTES
        WHEN return_items.created_at IS NULL THEN rent_to_purchase_orders.created_at + INTERVAL 330 MINUTES
      END AS item_transaction_date,
      NULLIF(rent_to_purchase_items.payment_details:payableAfterPaymentOffers.byCashPreTax::STRING, '') AS tto_pay,   -- [MIGRATED] json_extract_path_text → variant path
      NULLIF(rent_to_purchase_items.offers_snapshot:[1].code::STRING, '') AS code_2,                                  -- [MIGRATED] json_extract_array_element_text → :[1] path
      NULLIF(rent_to_purchase_items.offers_snapshot:[1].discountAmount::STRING, '')::DOUBLE AS code_2_discount,       -- [MIGRATED] ::float → ::DOUBLE
      CASE
        WHEN return_items.created_at IS NULL
         AND rent_to_purchase_orders.created_at IS NULL THEN NULL
        WHEN rent_to_purchase_orders.created_at IS NULL THEN 'return_item'
        WHEN return_items.created_at IS NULL THEN 'rent_to_purchase_item'
      END AS Transaction_type,
      return_items.updated_at + INTERVAL 330 MINUTES AS return_item_updated_at
    FROM furlenco_silver.order_management_systems_evolve.items
    LEFT JOIN furlenco_silver.order_management_systems_evolve.return_items
      ON items.id = return_items.item_id
     AND return_items.state != 'CANCELLED'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.rent_to_purchase_items
      ON rent_to_purchase_items.item_id = items.id
     AND POSITION('paid' IN LOWER(CAST(rent_to_purchase_items.payment_details AS STRING))) > 0   -- [MIGRATED] VARIANT cast to STRING for text search
    LEFT JOIN furlenco_silver.order_management_systems_evolve.rent_to_purchase_orders
      ON rent_to_purchase_orders.id = rent_to_purchase_items.rent_to_purchase_order_id
    WHERE items.vertical = 'FURLENCO_RENTAL'
      AND items.state != 'CANCELLED'
  ),

  base1 AS (
    SELECT
      base.item_id AS base_item_id,
      base.activation_date AS base_activation_date,
      base.user_id AS base_user_id,
      base.Transaction_id AS base_transaction_id,
      base.item_transaction_date AS base_item_transaction_date,
      base.Transaction_type AS base_transaction_type,
      dummy.*
    FROM base
    LEFT JOIN (SELECT * FROM base) AS dummy
      ON dummy.user_id = base.user_id
     AND dummy.activation_date < base.item_transaction_date
    WHERE base.Transaction_id IS NOT NULL
  ),

  fullorpartial AS (
    SELECT DISTINCT
      base_transaction_id,
      base_user_id,
      base_Transaction_type,
      CASE
        WHEN count(DISTINCT base_item_id) = count(
          DISTINCT CASE
            WHEN item_transaction_date >= base_item_transaction_date THEN item_id
            ELSE NULL
          END
        ) THEN 'FULL'
        ELSE 'PARTIAL'
      END AS VAR1
    FROM base1
    GROUP BY 1, 2, 3
    -- [MIGRATED] ORDER BY 1 DESC removed from CTE: no semantic effect, avoids an unnecessary sort
  ),

  ttto_base AS (
    SELECT DISTINCT
      i.id AS item_id,
      i.bundle_id,
      i.name AS item_name,
      return_item_state,
      i.composite_item_ID,
      i.pricing_details:basePrice::STRING AS base_price,                -- [MIGRATED] json_extract_path_text → variant path
      i.user_id,
      i.user_details:contactNo::STRING AS contact,
      i.user_details:name::STRING AS user_name,
      i.user_details:emailId::STRING AS emailId,
      i.user_details:displayId::STRING AS fur_id,
      i.activation_date + INTERVAL 330 MINUTES AS activation_date,
      CASE WHEN transaction_type = 'rent_to_purchase_item' THEN base.item_transaction_date
           ELSE i.pickup_date END AS pickup_date,
      i.is_migrated_for_evolve,
      i.is_autopay_enabled,
      'NA' AS ufdiscount_config,
      code_2 AS discount_coupon,
      base.item_transaction_date AS payment_date,
      CURRENT_DATE() AS charge_till_date,                               -- [MIGRATED] date(CURRENT_DATE) → CURRENT_DATE()
      base.item_transaction_date AS applicable_on,
      tto_pay::DOUBLE AS TTO_amount,
      base.Transaction_type AS Transaction_type,
      Transaction_id,
      VAR1 AS Transaction_type_detail,
      CURRENT_DATE() AS updated_at,
      return_item_updated_at
    FROM furlenco_silver.order_management_systems_evolve.items AS i
    LEFT JOIN base
      ON i.id = base.item_id
    LEFT JOIN fullorpartial
      ON fullorpartial.base_transaction_id = base.Transaction_id
     AND base.Transaction_type = base_Transaction_type
    WHERE Transaction_id IS NOT NULL
  ),

  base_ AS (
    SELECT
      i.id AS item_ids,
      i.user_id AS user_ids,
      i.order_id,
      i.activation_date AS activation_dates,
      t.*
    FROM furlenco_silver.order_management_systems_evolve.items AS i
    LEFT JOIN ttto_base AS t ON i.id = t.item_id
    WHERE 1=1
      AND i.vertical = 'FURLENCO_RENTAL'
      AND i.state <> 'CANCELLED'
  ),

  churn_counts AS (
    SELECT
      b1.item_ids,
      COUNT(DISTINCT b2.item_ids) AS later_items_count
    FROM base_ AS b1
    LEFT JOIN base_ AS b2
      ON b1.user_ids = b2.user_ids
     AND b1.item_ids <> b2.item_ids
     AND b2.activation_dates < b1.pickup_date
     AND (b2.pickup_date IS NULL OR b2.pickup_date > b1.pickup_date)
    GROUP BY b1.item_ids
  ),

  final_tagged AS (
    SELECT
      b.*,
      CASE
        WHEN b.pickup_date IS NULL THEN 'ACTIVE'
        WHEN cc.later_items_count > 0 THEN 'PARTIAL'
        ELSE 'FULL'
      END AS churn_flag
    FROM base_ AS b
    JOIN churn_counts AS cc ON b.item_ids = cc.item_ids
  ),

  ttto_base_mv AS (
    SELECT
      ft.*, rr.start_date, rr.end_date,
      rr.monetary_components:taxableAmount::DOUBLE
        / ROUND(DATEDIFF(day, rr.start_date, rr.end_date) / 30.40) AS taxable_amount,   -- [MIGRATED] date subtraction → DATEDIFF
      CAST(DATE_TRUNC('month', DATE_ADD(rr.end_date, 1)) AS DATE) AS revenue_loss_month, -- [MIGRATED] end_date + 1 → DATE_ADD
      DATE_ADD(rr.end_date, 1) AS revenue_loss_day,
      recognition_type,
      dense_rank() OVER (PARTITION BY accountable_entity_id, accountable_entity_type ORDER BY rr.start_date DESC) AS rnk,
      (recognised_at + INTERVAL 330 MINUTES) AS recognised_at
    FROM final_tagged AS ft
    LEFT JOIN furlenco_silver.furbooks_evolve.revenue_recognitions AS rr
      ON ft.item_ids = rr.accountable_entity_id AND rr.accountable_entity_type = 'ITEM'
    WHERE 1=1
      AND rr.state NOT IN ('CANCELLED','INVALIDATED')
      AND rr.deleted_at IS NULL    -- [RULE] added per data contract (deleted_at IS NULL on revenue_recognitions — ALWAYS); not in original Redshift proc
  ),

  aggregated_attachment AS (
    SELECT
      tm.*,
      at.id AS attachment_id,
      rr.monetary_components:taxableAmount::DOUBLE
        / ROUND(DATEDIFF(day, rr.start_date, rr.end_date) / 30.40) AS attachment_taxable_amount,
      at.pricing_details:basePrice::STRING AS attachment_base_price,
      dense_rank() OVER (PARTITION BY accountable_entity_id, accountable_entity_type ORDER BY rr.start_date DESC) AS attach_rnk,
      MAX(CASE WHEN CAST(payment_date AS DATE) > DATE_ADD(tm.start_date, -1)
                AND CAST(payment_date AS DATE) < DATE_ADD(tm.end_date, 2)
               THEN CAST(DATE_TRUNC('month', DATE_ADD(tm.end_date, 1)) AS DATE) END) OVER (PARTITION BY item_ids) AS loss_month,
      MAX(CASE WHEN CAST(payment_date AS DATE) > DATE_ADD(tm.start_date, -1)
                AND CAST(payment_date AS DATE) < DATE_ADD(tm.end_date, 2)
               THEN DATE_ADD(tm.end_date, 1) END) OVER (PARTITION BY item_ids) AS loss_day
    FROM ttto_base_mv AS tm
    LEFT JOIN furlenco_silver.order_management_systems_evolve.attachments AS at
      ON tm.composite_item_id = at.composite_item_id
    LEFT JOIN furlenco_silver.furbooks_evolve.revenue_recognitions AS rr
      ON at.id = rr.accountable_entity_id AND rr.accountable_entity_type = 'ATTACHMENT'
     AND rr.state NOT IN ('CANCELLED','INVALIDATED')
     AND rr.deleted_at IS NULL    -- [RULE] added per data contract; not in original Redshift proc
    WHERE 1=1
  ),

  item_vas AS (
    SELECT
      vas.entity_id, vas.entity_type,
      vas.start_date AS vas_item_start_date, vas.end_date AS vas_item_end_date,
      it.composite_item_id,
      rr.monetary_components:taxableAmount::DOUBLE AS taxableAmount,
      ROUND(DATEDIFF(day, vas.start_date, vas.end_date) / 30.60) AS tenures,
      dense_rank() OVER (PARTITION BY entity_id, entity_type ORDER BY vas.start_date DESC) AS item_vas_rnk
    FROM furlenco_silver.order_management_systems_evolve.items AS it
    LEFT JOIN furlenco_silver.order_management_systems_evolve.value_added_services AS vas
      ON vas.entity_id = it.id AND vas.entity_type = 'ITEM'
    LEFT JOIN furlenco_silver.furbooks_evolve.revenue_recognitions AS rr
      ON vas.id = rr.accountable_entity_id
    WHERE rr.accountable_entity_type = 'VALUE_ADDED_SERVICE'
      AND rr.deleted_at IS NULL    -- [RULE] added per data contract; not in original Redshift proc
      AND vas.type IN ('FURLENCO_CARE_PROGRAM')
      AND vas.state <> 'CANCELLED'
  ),

  attachment_vas AS (
    SELECT
      vas.entity_id, vas.entity_type,
      vas.start_date AS vas_attach_start_date, vas.end_date AS vas_attach_end_date,
      at.composite_item_id,
      rr.monetary_components:taxableAmount::DOUBLE AS attachment_taxableAmount,
      ROUND(DATEDIFF(day, vas.start_date, vas.end_date) / 30.60) AS at_tenures,
      dense_rank() OVER (PARTITION BY entity_id, entity_type ORDER BY vas.start_date DESC) AS attach_vas_rnk
    FROM furlenco_silver.order_management_systems_evolve.attachments AS at
    LEFT JOIN furlenco_silver.order_management_systems_evolve.value_added_services AS vas
      ON vas.entity_id = at.id AND vas.entity_type = 'ATTACHMENT'
    LEFT JOIN furlenco_silver.furbooks_evolve.revenue_recognitions AS rr
      ON vas.id = rr.accountable_entity_id
    WHERE rr.accountable_entity_type = 'VALUE_ADDED_SERVICE'
      AND rr.deleted_at IS NULL    -- [RULE] added per data contract; not in original Redshift proc
      AND vas.type IN ('FURLENCO_CARE_PROGRAM')
      AND vas.state <> 'CANCELLED'
  ),

  monthly_cte_item_attach AS (
    SELECT DISTINCT
      iv.*,
      (taxableAmount / nullif(tenures, 0)) AS monthly_item_taxable_vas,
      av.entity_id AS attach_id,
      av.attachment_taxableAmount, av.at_tenures, vas_attach_start_date, vas_attach_end_date,
      (attachment_taxableAmount / nullif(at_tenures, 0)) AS monthly_attachment_taxable_vas,
      attach_vas_rnk
    FROM item_vas AS iv
    LEFT JOIN attachment_vas AS av
      ON iv.composite_item_id = av.composite_item_id
    WHERE 1=1
      AND (attach_vas_rnk = 1 OR attach_vas_rnk IS NULL)   -- [MIGRATED] postfix isnull → IS NULL
      AND item_vas_rnk = 1
  ),

  total_taxable_amount AS (
    SELECT DISTINCT
      tv.*,
      taxableAmount AS item_taxableamount_vas, tenures, vas_item_start_date, vas_item_end_date, monthly_item_taxable_vas,
      attachment_taxableAmount AS attachment_taxableamount_vas, at_tenures, vas_attach_start_date, vas_attach_end_date, monthly_attachment_taxable_vas,
      CASE WHEN tv.end_date <= vas_item_end_date THEN monthly_item_taxable_vas END AS vas_item_tx,
      CASE WHEN tv.end_date <= vas_attach_end_date THEN monthly_attachment_taxable_vas END AS vas_attach_tx
    FROM aggregated_attachment AS tv
    LEFT JOIN monthly_cte_item_attach AS mta
      ON tv.item_ids = mta.entity_id AND entity_type = 'ITEM'
    WHERE 1=1
      AND attach_rnk = 1
  ),

  all_vas_and_item_attach AS (
    SELECT
      *,
      (taxable_amount::DOUBLE + COALESCE(attachment_taxable_amount, 0)::DOUBLE
       + COALESCE(vas_item_tx, 0)::DOUBLE + COALESCE(vas_attach_tx, 0)::DOUBLE) AS total_taxable_amount_value
    FROM total_taxable_amount
  ),

  rental_final_output AS (
    SELECT
      * EXCEPT (item_id, loss_month, revenue_loss_month, loss_day, revenue_loss_day),   -- [MIGRATED] EXCLUDE → EXCEPT
      DATEDIFF(day, end_date, CAST(pickup_date AS DATE)) AS post_ctd_pickup_days,        -- [MIGRATED] date subtraction → DATEDIFF
      COALESCE(loss_day, revenue_loss_day) AS revenue_loss_day,
      COALESCE(loss_month, revenue_loss_month) AS revenue_loss_month,
      total_taxable_amount_value::DOUBLE
        / CAST(DAY(LAST_DAY(revenue_loss_month)) AS DOUBLE) AS per_day_revenue           -- [MIGRATED] DATE_PART(day,...) → DAY(); binds to source column, matching Redshift alias rules
    FROM all_vas_and_item_attach
    WHERE 1=1
      AND churn_flag <> 'ACTIVE'
  )

SELECT
  rn.item_ids, rn.user_ids, rn.order_id, rn.activation_dates,
  rn.bundle_id, rn.item_name, rn.return_item_state, rn.composite_item_ID,
  rn.base_price, rn.user_id, rn.contact, rn.user_name,
  rn.emailId, rn.fur_id, rn.activation_date, rn.pickup_date,
  rn.is_migrated_for_evolve, rn.is_autopay_enabled, rn.ufdiscount_config, rn.discount_coupon,
  rn.payment_date, rn.charge_till_date, rn.applicable_on, rn.TTO_amount,
  rn.Transaction_type, rn.Transaction_id, rn.Transaction_type_detail, rn.updated_at,
  rn.return_item_updated_at, rn.churn_flag, rn.start_date, rn.end_date,
  rn.taxable_amount, rn.recognition_type, rn.rnk, rn.recognised_at,
  rn.attachment_id, rn.attachment_taxable_amount, rn.attach_rnk, rn.attachment_base_price,
  rn.tenures, rn.vas_item_start_date, rn.vas_item_end_date, rn.monthly_item_taxable_vas,
  rn.at_tenures, rn.vas_attach_start_date, rn.vas_attach_end_date, rn.monthly_attachment_taxable_vas,
  rn.total_taxable_amount_value, rn.post_ctd_pickup_days, rn.revenue_loss_day, rn.revenue_loss_month,
  rn.per_day_revenue, rn.vas_item_tx AS item_taxableamount_vas, rn.vas_attach_tx AS attachment_taxableamount_vas,
  it.charged_till_date AS charged_till_date_from_item,
  CURRENT_TIMESTAMP() + INTERVAL '330 MINUTES' AS refreshed_at    -- [IST] UTC → IST
FROM rental_final_output AS rn
LEFT JOIN furlenco_silver.order_management_systems_evolve.items AS it
  ON rn.item_ids = it.id


In [0]:

-- ============================================================================
-- BLOCK 2 — VALIDATE STAGING (4 checks + audit log)
-- Any RAISE_ERROR fails this block and stops the notebook — swap is blocked.
--   1. Empty check     — staging must have at least 1 row
--   2. Row count check — staging must have >= 90% of rows vs live table
--   3. Schema check    — all columns in live must exist in staging
--   4. Null check      — key business columns must not be entirely NULL
-- ============================================================================

-- CHECK 1: Staging must not be empty
SELECT
  CASE
    WHEN COUNT(*) = 0
    THEN RAISE_ERROR('VALIDATION FAILED: Staging table is empty. Swap blocked.')
  END
FROM furlenco_analytics.materialized_tables.rental_churn_query_staging;

-- FIRST-RUN BOOTSTRAP: if the live table does not exist yet, create it from
-- staging so Check 2, Check 3 and the audit log have a table to compare
-- against. No-op on every subsequent run (IF NOT EXISTS). Runs after the
-- empty check so an empty staging can never become the live table.
CREATE TABLE IF NOT EXISTS furlenco_analytics.materialized_tables.rental_churn_query
  DEEP CLONE furlenco_analytics.materialized_tables.rental_churn_query_staging;

-- CHECK 2: Staging row count must be >= 90% of live table row count
SELECT
  CASE
    WHEN staging_count < (live_count * 0.90)
    THEN RAISE_ERROR(
      CONCAT(
        'VALIDATION FAILED: Row count check failed. ',
        'Staging rows: ', CAST(staging_count AS STRING),
        ' | Live rows: ',  CAST(live_count    AS STRING),
        ' | Threshold (90%): ', CAST(CAST(live_count * 0.90 AS BIGINT) AS STRING),
        '. Swap blocked.'
      )
    )
  END
FROM (
  SELECT
    (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.rental_churn_query_staging) AS staging_count,
    (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.rental_churn_query) AS live_count
);

-- CHECK 3: Schema match — no columns should be missing from staging vs live
SELECT
  CASE
    WHEN COUNT(*) > 0
    THEN RAISE_ERROR(
      CONCAT(
        'VALIDATION FAILED: Schema mismatch detected. ',
        'Columns in live but missing from staging: ',
        ARRAY_JOIN(COLLECT_LIST(column_name), ', '),
        '. Swap blocked.'
      )
    )
  END
FROM (
  SELECT column_name
  FROM information_schema.columns
  WHERE table_catalog = 'furlenco_analytics'
    AND table_schema  = 'materialized_tables'
    AND table_name    = 'rental_churn_query'
  EXCEPT
  SELECT column_name
  FROM information_schema.columns
  WHERE table_catalog = 'furlenco_analytics'
    AND table_schema  = 'materialized_tables'
    AND table_name    = 'rental_churn_query_staging'
);

-- CHECK 4: Key business columns must not be entirely NULL
SELECT
  CASE
    WHEN item_ids_nulls = total_rows THEN RAISE_ERROR('VALIDATION FAILED: item_ids is entirely NULL in staging.')
    WHEN user_ids_nulls = total_rows THEN RAISE_ERROR('VALIDATION FAILED: user_ids is entirely NULL in staging.')
    WHEN fur_id_nulls = total_rows THEN RAISE_ERROR('VALIDATION FAILED: fur_id is entirely NULL in staging.')
    WHEN total_taxable_amount_value_nulls = total_rows THEN RAISE_ERROR('VALIDATION FAILED: total_taxable_amount_value is entirely NULL in staging.')
  END
FROM (
  SELECT
    COUNT(*)                                       AS total_rows,
    COUNT_IF(item_ids IS NULL)                     AS item_ids_nulls,
    COUNT_IF(user_ids IS NULL)                     AS user_ids_nulls,
    COUNT_IF(fur_id IS NULL)                       AS fur_id_nulls,
    COUNT_IF(total_taxable_amount_value IS NULL)   AS total_taxable_amount_value_nulls
  FROM furlenco_analytics.materialized_tables.rental_churn_query_staging
);

-- All checks passed — log staging stats for the Workflow run audit trail
SELECT
  'VALIDATION PASSED'                          AS status,
  (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.rental_churn_query_staging) AS staging_row_count,
  (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.rental_churn_query) AS live_row_count,
  current_timestamp() + INTERVAL '330 MINUTES'   AS validated_at_ist;


In [0]:

-- ============================================================================
-- BLOCK 3 — ATOMIC SWAP + CLEANUP (ZERO-DOWNTIME)
-- CREATE OR REPLACE ... DEEP CLONE is a single atomic Delta commit: readers
-- querying the table (or the gold view) during the swap see the previous
-- snapshot until the commit lands. The live table is never dropped.
-- ============================================================================

-- STEP 1: Optimize staging BEFORE the swap, so the clone copies
-- already-optimized files and live is never rewritten post-swap.
OPTIMIZE furlenco_analytics.materialized_tables.rental_churn_query_staging
  ZORDER BY (churn_flag, Transaction_type);

-- STEP 2: Atomic replace — live table stays queryable throughout
CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.rental_churn_query
  DEEP CLONE furlenco_analytics.materialized_tables.rental_churn_query_staging;

-- STEP 3: Stamp refresh metadata (IST) on the live table.
-- EXECUTE IMMEDIATE is required — SET TBLPROPERTIES does not accept
-- function calls like current_timestamp() directly in Databricks.
EXECUTE IMMEDIATE
  CONCAT(
    "ALTER TABLE furlenco_analytics.materialized_tables.rental_churn_query SET TBLPROPERTIES (",
    "'last_refreshed_at_ist' = '", string(current_timestamp() + INTERVAL '330 MINUTES'), "', ",
    "'refresh_type' = 'full_refresh', ",
    "'refresh_schedule' = 'configured_in_workflow')"
  );

-- STEP 4: Drop staging — pure cleanup. After DEEP CLONE the live table owns
-- its own copy of the data, so dropping staging cannot affect live queries.
DROP TABLE IF EXISTS furlenco_analytics.materialized_tables.rental_churn_query_staging;

-- Final confirmation — output appears in the Workflow run log
SELECT
  'SWAP SUCCESSFUL'                            AS status,
  COUNT(*)                                     AS live_row_count,
  current_timestamp() + INTERVAL '330 MINUTES'   AS swapped_at_ist
FROM furlenco_analytics.materialized_tables.rental_churn_query;


In [0]:

-- ============================================================================
-- BLOCK 4 — ONE-TIME GOLD VIEW SETUP (REFERENCE ONLY — ALREADY EXECUTED)
-- The view furlenco_gold.analytics.rental_churn_query was created on 2026-06-10.
-- Kept commented out so the scheduled Workflow run does not re-execute DDL.
-- Uncomment and run ONCE manually only if the view ever needs recreating.
-- ============================================================================

-- CREATE OR REPLACE VIEW furlenco_gold.analytics.rental_churn_query
-- AS SELECT * FROM furlenco_analytics.materialized_tables.rental_churn_query;

